# 03 · Animations and games

`board.animate(frame_fn, duration, fps)` calls `frame_fn(t)` again and again
(`t` = seconds since the start) and shows whatever dictionary it returns.

**Reality check:** Bluetooth Low Energy is slow. A frame with all 165 LEDs is ~500 bytes,
sent in 20-byte pieces, so expect **a few frames per second**, not 60.
Frames with fewer LEDs are faster.

In [ ]:
# Make the repository root importable when the notebook runs from examples/
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from boardlights import Board, Wall, scan

In [ ]:
# --- Configure your wall here ------------------------------------------------
# IICW default: Kilter Homewall (layout 8), 7x10 Mainline LED kit (size 18, 165 LEDs).
# Not sure? Run `python -m boardlights configs` to list all layout/size ids.
LAYOUT_ID = 8
SIZE_ID = 18
BOARD_ADDRESS = None   # e.g. "FC:0C:87:2B:53:25"; None = connect to the closest board

In [ ]:
import colorsys, math, random

wall = Wall(LAYOUT_ID, SIZE_ID)
holds = wall.unique_holds()
min_x, max_x, min_y, max_y = wall.bounds()

def hsv(h, s=1.0, v=1.0):
    r, g, b = colorsys.hsv_to_rgb(h % 1.0, s, v)
    return (int(r * 255), int(g * 255), int(b * 255))

board = await Board.connect(address=BOARD_ADDRESS)

## Rainbow that scrolls up the wall

In [ ]:
def rainbow(t):
    return {h.position: hsv((h.y - min_y) / (max_y - min_y) + t * 0.3) for h in holds}

frames = await board.animate(rainbow, duration=15, fps=4)
print(frames, "frames sent")

## A horizontal line sweeping up and down
Only one row is lit, so frames are small and fast.

In [ ]:
rows = sorted({h.y for h in holds})

def sweep(t):
    i = int(t * 4) % (2 * len(rows) - 2)
    y = rows[i if i < len(rows) else 2 * len(rows) - 2 - i]
    return {h.position: "00FFFF" for h in holds if h.y == y}

await board.animate(sweep, duration=10, fps=8)

## Sparkle

In [ ]:
def sparkle(t):
    return {h.position: hsv(random.random()) for h in random.sample(holds, 15)}

await board.animate(sparkle, duration=10, fps=5)

## Game: reaction trainer
A random hold lights up; touch it and press Enter. How fast are you?
(Run this cell, then stand at the wall with someone at the keyboard, or use a wireless keyboard.)

In [ ]:
import time
reachable = [h for h in holds if h.y < 110]   # stay within reach
times = []
for round_ in range(5):
    target = random.choice(reachable)
    await board.show({target.position: "FF0000"})
    start = time.monotonic()
    input(f"Round {round_ + 1}: press Enter when you touch the red hold")
    times.append(time.monotonic() - start)
    await board.show({target.position: "00FF00"})
    time.sleep(0.5)
print("Times:", [f"{t:.2f}s" for t in times], "average", f"{sum(times) / len(times):.2f}s")

## Ideas to build next
* Snake or Tetris on the wall (`wall.nearest(x, y)` turns grid coordinates into holds)
* A "follow the leader" game: one player adds a hold each turn
* Show a countdown timer for hangboard-style intervals
* Light the holds of a climb one by one in order, from bottom to top

In [ ]:
await board.clear()
await board.disconnect()